In [0]:
# =========================================================
# BRONZE LAYER — TRAFFIC DATA INGESTION
# =========================================================
# Sources  : AWS S3 (4 parquet + 1 csv)
# Target   : traffic_catalog.bronze (Delta tables)
# Logging  : traffic_catalog.logs.error_log (matches silver/gold)
# Nulls    : injected at ~1% rate to simulate real data issues
# =========================================================

import logging
import uuid
import traceback
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("traffic_bronze_ingestion")

# =========================================================
# ERROR LOG SCHEMA (matches silver + gold layer)
# =========================================================

ERROR_LOG_SCHEMA = StructType([
    StructField("log_id",          StringType(),    False),
    StructField("pipeline_run_id", StringType(),    False),
    StructField("layer",           StringType(),    False),
    StructField("table_name",      StringType(),    False),
    StructField("step",            StringType(),    False),
    StructField("error_type",      StringType(),    True),
    StructField("error_message",   StringType(),    True),
    StructField("stack_trace",     StringType(),    True),
    StructField("rows_expected",   LongType(),      True),
    StructField("rows_written",    LongType(),      True),
    StructField("is_critical",     BooleanType(),   False),
    StructField("logged_at",       TimestampType(), False),
])

RUN_LOG_SCHEMA = StructType([
    StructField("run_id",        StringType(),    False),
    StructField("layer",         StringType(),    False),
    StructField("started_at",    TimestampType(), False),
    StructField("ended_at",      TimestampType(), True),
    StructField("status",        StringType(),    False),
    StructField("tables_ok",     IntegerType(),   True),
    StructField("tables_failed", IntegerType(),   True),
    StructField("notes",         StringType(),    True),
])

# =========================================================
# PIPELINE LOGGER (matches silver + gold)
# =========================================================

class BronzeLogger:
    def __init__(self, spark):
        self.spark         = spark
        self.run_id        = str(uuid.uuid4())
        self.tables_ok     = 0
        self.tables_failed = 0
        self.started_at    = datetime.now()

        spark.sql("CREATE SCHEMA IF NOT EXISTS traffic_catalog.logs")
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.error_log (
                log_id STRING, pipeline_run_id STRING, layer STRING,
                table_name STRING, step STRING, error_type STRING,
                error_message STRING, stack_trace STRING,
                rows_expected LONG, rows_written LONG,
                is_critical BOOLEAN, logged_at TIMESTAMP
            ) USING DELTA
        """)
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.pipeline_run_log (
                run_id STRING, layer STRING, started_at TIMESTAMP,
                ended_at TIMESTAMP, status STRING,
                tables_ok INT, tables_failed INT, notes STRING
            ) USING DELTA
        """)
        self._write_run("RUNNING", "Bronze ingestion started")
        logger.info(f"[BRONZE] run_id: {self.run_id} | started: {self.started_at}")

    def log_error(self, table, step, error,
                  is_critical=False, rows_expected=None, rows_written=None):
        row = {
            "log_id":          str(uuid.uuid4()),
            "pipeline_run_id": self.run_id,
            "layer":           "BRONZE",
            "table_name":      table,
            "step":            step.upper(),
            "error_type":      type(error).__name__,
            "error_message":   str(error)[:2000],
            "stack_trace":     traceback.format_exc()[:4000],
            "rows_expected":   rows_expected,
            "rows_written":    rows_written,
            "is_critical":     is_critical,
            "logged_at":       datetime.now(),
        }
        self.spark.createDataFrame([row], ERROR_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.error_log")
        lvl = "CRITICAL" if is_critical else "WARNING"
        logger.error(f"  [{lvl}] {table}.{step} — {type(error).__name__}: {str(error)[:150]}")

    def log_success(self, table, rows):
        self.tables_ok += 1
        logger.info(f"  [OK] {table} — {rows:,} rows written")

    def log_failure(self, table):
        self.tables_failed += 1

    def finish(self):
        status = "FAILED" if self.tables_failed > 0 else "SUCCESS"
        self._write_run(status,
                        f"{self.tables_ok} ok / {self.tables_failed} failed")
        logger.info(
            f"\n[BRONZE] {status} | ok: {self.tables_ok} | "
            f"failed: {self.tables_failed} | run_id: {self.run_id}"
        )
        return status

    def _write_run(self, status, notes=None):
        row = {
            "run_id":        self.run_id,
            "layer":         "BRONZE",
            "started_at":    self.started_at,
            "ended_at":      datetime.now() if status != "RUNNING" else None,
            "status":        status,
            "tables_ok":     self.tables_ok,
            "tables_failed": self.tables_failed,
            "notes":         notes,
        }
        self.spark.createDataFrame([row], RUN_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.pipeline_run_log")

# =========================================================
# S3 DATASET CONFIGURATION
# =========================================================
# 4 Parquet files + 1 CSV — all from AWS S3
# Update paths if new files arrive daily in S3

DATASETS = {
    "incident": {
        "path":   "s3://traffic-parquet-bucket/incidents/2026/03/12/09/"
                  "incident-firehose-2-2026-03-12-09-08-08-"
                  "34467de8-922d-46e1-9036-49b96a8db130.parquet",
        "format": "parquet",
        "expected_cols": [
            "event_id", "sensor_id", "event_timestamp", "incident_id",
            "incident_type", "incident_severity", "incident_duration",
            "vehicles_affected", "lane_blocked", "response_time"
        ]
    },
    "sensor": {
        "path":   "s3://traffic-parquet-bucket/sensor/"
                  "traffic_sensor_stream_raw_v2.csv",
        "format": "csv",
        "expected_cols": [
            "event_id", "sensor_id", "event_timestamp", "location_id",
            "lane_id", "vehicle_count", "avg_speed", "traffic_density",
            "occupancy_rate", "congestion_score"
        ]
    },
    "speed": {
        "path":   "s3://traffic-parquet-bucket/speed/2026/03/12/08/"
                  "speed-firehose-2-2026-03-12-08-50-22-"
                  "43c1fa8d-82a3-49d3-96d3-2475522740e7.parquet",
        "format": "parquet",
        "expected_cols": [
            "event_id", "sensor_id", "event_timestamp", "speed_limit",
            "avg_speed", "min_speed", "max_speed", "speed_variance",
            "speed_violation_count", "speed_index"
        ]
    },
    "vehicle": {
        "path":   "s3://traffic-parquet-bucket/vehicle/2026/03/12/08/"
                  "vehicle-firehose-2-2026-03-12-08-58-42-"
                  "73357f91-13a0-4424-b927-7a06d37cde00.parquet",
        "format": "parquet",
        "expected_cols": [
            "event_id", "sensor_id", "event_timestamp", "vehicle_type",
            "vehicle_count", "vehicle_flow_rate", "vehicle_density",
            "queue_length", "lane_utilization", "vehicle_delay"
        ]
    },
    "signal": {
        "path":   "s3://traffic-parquet-bucket/signal/2026/03/12/08/"
                  "signal-firehose-2-2026-03-12-08-50-02-"
                  "f4c12218-5c2f-475a-b0ab-e55ee861a559.parquet",
        "format": "parquet",
        "expected_cols": [
            "event_id", "sensor_id", "event_timestamp", "signal_id",
            "signal_cycle_time", "green_time", "red_time",
            "signal_wait_time", "queue_length", "signal_efficiency"
        ]
    }
}

BRONZE_SCHEMA = "traffic_catalog.bronze"

# =========================================================
# BRONZE INGESTION
# =========================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
bronze_log = BronzeLogger(spark)

logger.info("\n" + "=" * 50)
logger.info("BRONZE INGESTION — READING FROM S3")
logger.info("=" * 50)

for table_name, config in DATASETS.items():

    path          = config["path"]
    file_format   = config["format"]
    expected_cols = config["expected_cols"]
    full_table    = f"{BRONZE_SCHEMA}.{table_name}"

    # ── Step 1: Read from S3 ─────────────────────────────
    try:
        if file_format == "parquet":
            df = spark.read.format("parquet") \
                .option("recursiveFileLookup", "true") \
                .load(path)
        elif file_format == "csv":
            df = spark.read.format("csv") \
                .option("header",      "true") \
                .option("inferSchema", "true") \
                .load(path)
        else:
            raise ValueError(f"Unsupported format: {file_format}")

        # Normalise column names to lowercase
        df = df.toDF(*[c.lower() for c in df.columns])

        logger.info(f"  [READ] {table_name} — {file_format} from S3 "
                    f"— {df.count():,} rows, {len(df.columns)} cols")

    except Exception as e:
        bronze_log.log_error(table_name, "READ", e, is_critical=True)
        bronze_log.log_failure(table_name)
        logger.error(f"  [FAIL] {table_name} read from S3 — skipping")
        continue

    # ── Step 2: Schema validation ────────────────────────
    try:
        actual_cols  = set(df.columns)
        missing_cols = set(expected_cols) - actual_cols
        if missing_cols:
            raise ValueError(
                f"Missing expected columns: {missing_cols}"
            )
        logger.info(f"  [SCHEMA] {table_name} — all {len(expected_cols)} columns present")

    except Exception as e:
        bronze_log.log_error(table_name, "SCHEMA_VALIDATE", e,
                             is_critical=False)  # Warning only — proceed anyway

    # ── Step 3: Row count check ──────────────────────────
    try:
        row_count = df.count()
        if row_count == 0:
            raise ValueError(f"{table_name} has 0 rows — S3 file may be empty")
        logger.info(f"  [COUNT] {table_name} — {row_count:,} rows")

    except Exception as e:
        bronze_log.log_error(table_name, "ROW_COUNT", e, is_critical=True)
        bronze_log.log_failure(table_name)
        continue

    # ── Step 4: Write to Delta ───────────────────────────
    try:
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(full_table)

        written = spark.table(full_table).count()
        bronze_log.log_success(table_name, written)
        logger.info(f"  [WRITTEN] {full_table} — {written:,} rows")

    except Exception as e:
        bronze_log.log_error(table_name, "WRITE", e, is_critical=True,
                             rows_expected=row_count, rows_written=0)
        bronze_log.log_failure(table_name)

# =========================================================
# NULL VALUE INJECTION
# Simulates real-world data quality issues at ~1% rate
# Matches original bronze code behaviour
# =========================================================

logger.info("\n" + "=" * 50)
logger.info("NULL VALUE INJECTION")
logger.info("=" * 50)

null_injection_config = {
    "incident": [
        ("lane_blocked",      "string"),
        ("vehicles_affected", "long"),
    ],
    "signal": [
        ("signal_efficiency", "double"),
        ("queue_length",      "double"),
    ],
    "speed": [
        ("speed_variance",        "double"),
        ("speed_violation_count", "long"),
    ],
    "vehicle": [
        ("vehicle_count", "long"),
        ("vehicle_delay", "double"),
    ],
    "sensor": [
        ("avg_speed",      "double"),
        ("occupancy_rate", "double"),
    ],
}

for table_name, columns in null_injection_config.items():
    full_table = f"{BRONZE_SCHEMA}.{table_name}"
    try:
        df = spark.table(full_table)

        for col_name, col_type in columns:
            df = df.withColumn(
                col_name,
                F.when(F.rand() < 0.01, F.lit(None).cast(col_type))
                 .otherwise(F.col(col_name))
            )

        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table)

        null_counts = {
            c: spark.table(full_table)
                    .filter(F.col(c).isNull()).count()
            for c, _ in columns
        }
        logger.info(f"  [NULL] {table_name} — injected nulls: {null_counts}")

    except Exception as e:
        bronze_log.log_error(table_name, "NULL_INJECTION", e,
                             is_critical=False)

logger.info("NULL injection complete")

# =========================================================
# FINAL VALIDATION
# =========================================================

logger.info("\n" + "=" * 50)
logger.info("BRONZE VALIDATION SUMMARY")
logger.info("=" * 50)

all_ok = True
for table_name in DATASETS.keys():
    full_table = f"{BRONZE_SCHEMA}.{table_name}"
    try:
        df    = spark.table(full_table)
        rows  = df.count()
        cols  = len(df.columns)
        nulls = sum(
            df.filter(F.col(c).isNull()).count()
            for c in df.columns
        )
        logger.info(
            f"  {table_name:<12} | {rows:>7,} rows | "
            f"{cols} cols | {nulls} nulls injected"
        )
    except Exception as e:
        logger.error(f"  {table_name} — validation failed: {e}")
        all_ok = False

# =========================================================
# FINISH
# =========================================================

final_status = bronze_log.finish()

logger.info("\n" + "=" * 50)
logger.info("BRONZE INGESTION COMPLETE")
logger.info("=" * 50)
logger.info(f"Status  : {final_status}")
logger.info(f"Run ID  : {bronze_log.run_id}")
logger.info(
    f"Query errors: SELECT * FROM traffic_catalog.logs.error_log "
    f"WHERE pipeline_run_id = '{bronze_log.run_id}'"
)